# Notebook B — Offline Policy Training (Stage A) + GNN Fine-tune (Stage B, optional)

## Setup (carried over from your already-run Notebook A)
These 6 cells are copied verbatim from the Notebook A you already ran successfully
(same PAT-based private clone, same `INPUT` dataset path, same config, same frozen
model checkpoint, same inline RL helper classes). You do **not** need to re-run
Notebook A first in this session — just run these cells here, top to bottom, then
continue into Stage A training below.

**One cleanup applied:** the old dead public `git clone` line in the "mount data"
cell has been removed — your Cell 2 (PAT clone) already creates the repo folder, so
that line never actually ran anyway; this just removes the leftover.

## Setup (from your already-run Notebook A)

In [ ]:
import torch, sys, subprocess
print("torch:", torch.__version__, "| python:", sys.version.split()[0], "| cuda:", torch.cuda.is_available())
TORCH = torch.__version__.split("+")[0]
def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(args))
pip("torch-geometric==2.6.1", "torch-scatter", "torch-sparse", "torch-cluster",
    "-f", f"https://data.pyg.org/whl/torch-{TORCH}+cpu.html")
pip("pyyaml", "scipy", "pandas", "matplotlib", "seaborn")
print("deps OK")

In [ ]:
import os
import subprocess

REPO_DIR = "/kaggle/working/cosmic-net"
# Audited implementation lives on fork fix/rl-pruning-symmetry (upstream main
# lags the audited commits); pin both so execution matches the reviewed code.
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "fix/rl-pruning-symmetry"

# Read GitHub PAT from Kaggle Secrets
# If using kaggle_secrets:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
GITHUB_PAT = user_secrets.get_secret("GITHUB_PAT")

# Clone only if repo doesn't already exist
if not os.path.exists(REPO_DIR):
    auth_url = REPO_URL.replace(
        "https://",
        f"https://x-access-token:{GITHUB_PAT}@"
    )

    # SECURITY: never let the token reach the notebook output. A failed
    # subprocess.check_call raises CalledProcessError whose message embeds the
    # full command line (i.e. the token) — re-raise a sanitized error instead.
    try:
        subprocess.check_call([
            "git", "clone",
            "--branch", REPO_BRANCH,
            auth_url,
            REPO_DIR
        ])
    except subprocess.CalledProcessError:
        raise RuntimeError(
            f"git clone failed for {REPO_URL} (token redacted) — check the PAT "
            "secret, the repo URL, and Kaggle internet access."
        ) from None
    # The clone writes the token into .git/config (remote origin URL) — remove it.
    subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])

# FIX (Sep 2026 rewire): the audited RL fixes live on fix/rl-pruning-symmetry.
# A cached REPO_DIR from an older Kaggle run would otherwise silently execute
# stale pre-fix code — always fetch + check out the branch, then scrub the PAT
# from the remote URL again (fetch writes it back into .git/config).
auth_fetch = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_PAT}@")
try:
    subprocess.check_call(["git", "-C", REPO_DIR, "fetch", auth_fetch, REPO_BRANCH])
except subprocess.CalledProcessError:
    raise RuntimeError(f"git fetch failed for {REPO_URL} (token redacted) — check the PAT secret and Kaggle internet access.") from None
subprocess.check_call(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, "FETCH_HEAD"])
subprocess.check_call(["git", "-C", REPO_DIR, "remote", "set-url", "origin", REPO_URL])
print("branch:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip(),
      "| HEAD:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())


os.chdir(REPO_DIR)
print(os.listdir("."))

In [ ]:
# Mount uploaded data (repo already cloned privately in the previous cell)
import subprocess, os, shutil
os.chdir("/kaggle/working/cosmic-net")
print(os.listdir("."))
# Upload tng100_clustered.csv + best_model_augmented.pt as a Kaggle dataset named "cosmicnet-data"
INPUT = "/kaggle/input/datasets/nealsalian/cosmicnet-data"
os.makedirs("data/raw", exist_ok=True)
shutil.copy(f"{INPUT}/tng100_clustered.csv", "data/raw/tng100_clustered.csv")
os.makedirs("kaggle", exist_ok=True)
if os.path.exists(f"{INPUT}/best_model_augmented.pt"):
    shutil.copy(f"{INPUT}/best_model_augmented.pt", "kaggle/best_model_augmented.pt")
print("data staged:", os.path.getsize("data/raw/tng100_clustered.csv")/1e6, "MB")


In [ ]:
#Load config, force CPU-safe worker settings
import sys, yaml, torch, numpy as np
sys.path.insert(0, ".")
with open("config/config.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["data"]["source"] = "tng"
cfg["data"]["num_workers"] = 0         
cfg["data"]["batch_size"] = 16
cfg["model"]["mc_samples"] = 30
cfg["rls"] = {
    "policy_hidden": 64, "lr": 0.001, "entropy_coef": 0.01, "value_coef": 0.5,
    "epochs": 60, "batch_size": 32,
    "target_sparsity_start": 0.9, "target_sparsity_end": 0.4,
    "sparsity_anneal_epochs": 40,
    "w_acc": 1.0, "w_sp": 0.5, "w_conn": 1.0, "w_virial": 1.0, "w_unc": 0.5,
    "virial_anneal_start_epoch": 10, "min_keep_frac": 0.1, "seed": 42,
    # TTA ("RL at inference") — tune on VAL split only, then freeze
    "tta_lr": 1e-4, "tta_steps": 10, "tta_mc_samples": 15, "tta_patience": 3,
    "tta_target_sparsity": 0.5,
}

# Sep 2026 fixes (wired to rls/ below): headline sparsity path + guarded
# Stage B + TTA trust region. setdefault so a future config.yaml switch wins.
# (The headline cell below sets sparsity_mode="topk_scheduled" explicitly;
# the "penalty" ablation lives in the appendix cell at the end.)
cfg["rls"].setdefault("sparsity_mode", "penalty")
cfg["rls"].setdefault("stageb_epochs", 10)
cfg["rls"].setdefault("stageb_lr", 1e-4)
cfg["rls"].setdefault("stageb_patience", 3)
cfg["rls"].setdefault("stageb_full_tol", 0.02)
cfg["rls"].setdefault("stageb_augment", "policy")
# TTA trust-region placeholders — NOT tuned; sweep on VAL on Kaggle (see the
# tuning markdown before the TTA cells in Notebook D).
cfg["rls"].setdefault("tta_kl_coef", 0.05)
cfg["rls"].setdefault("tta_lr_decay", 0.9)
cfg["rls"].setdefault("tta_val_tol", 0.02)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| source:", cfg["data"]["source"])

In [ ]:
# PROVENANCE (recording only - no computation is affected; no secrets are read)
import hashlib, json, platform, subprocess, time

def _md5_of(path, _blk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(_blk), b""):
            h.update(chunk)
    return h.hexdigest()

def _file_info(path):
    import os
    ex = os.path.exists(path)
    return {"path": path, "exists": ex,
            "size_bytes": os.path.getsize(path) if ex else None,
            "md5": _md5_of(path) if ex else None}

def _git_info(*args):
    try:
        return subprocess.check_output(["git", "-C", REPO_DIR, *args],
                                       text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

def record_provenance(stage, extra=None):
    """Append one stage entry to outputs/rls/provenance.json (keyed by stage,
    so A/B/B_policy/C/D entries coexist and are never silently overwritten)."""
    import os
    entry = {
        "stage": stage,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "repo": {"head": _git_info("rev-parse", "HEAD"),
                 "branch": _git_info("rev-parse", "--abbrev-ref", "HEAD"),
                 "describe": _git_info("describe", "--always")},
        "dataset": {"tng_csv": _file_info("data/raw/tng100_clustered.csv"),
                    "checkpoint": _file_info(f"{INPUT}/best_model_augmented.pt")},
        "seed": cfg.get("seed"),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": {"available": torch.cuda.is_available(),
                 "version": str(torch.version.cuda),
                 "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
        "config_used": {"data": cfg.get("data"), "graph": cfg.get("graph"),
                        "model": cfg.get("model"), "rls": cfg.get("rls")},
    }
    if extra:
        entry.update(extra)
    os.makedirs("outputs/rls", exist_ok=True)
    prov = {}
    try:
        with open("outputs/rls/provenance.json") as f:
            prov = json.load(f)
    except Exception:
        pass
    prov[stage] = entry
    with open("outputs/rls/provenance.json", "w") as f:
        json.dump(prov, f, indent=2)
    print(f"[provenance] stage '{stage}' -> outputs/rls/provenance.json "
          f"(HEAD={entry['repo']['head']}, csv md5={entry['dataset']['tng_csv']['md5']})")

record_provenance("B")

In [ ]:
#Load halos, build graphs, split
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder, build_dataloaders
loader = get_loader(cfg)
halos = loader.load()
print("total halos:", len(halos), "| split", loader.split_data.__name__ if False else "")
train_halos, val_halos, test_halos = loader.split_data()
print(f"train={len(train_halos)} val={len(val_halos)} test={len(test_halos)}")
torch.manual_seed(cfg["seed"]); np.random.seed(cfg["seed"])   # AFTER split_data (it resets RNG)
train_loader, val_loader, test_loader = build_dataloaders(cfg, train_halos, val_halos, test_halos)
b0 = next(iter(test_loader))
print("graph:", b0.x.shape, b0.edge_index.shape, b0.edge_attr.shape, "y:", b0.y.shape)

In [ ]:
# Load frozen backbone and verify it predicts
from model.model import load_model
import torch
import torch_geometric.data as pyg_data

ckpt = "/kaggle/input/datasets/nealsalian/cosmicnet-data/best_model_augmented.pt"

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------
gnn = load_model(ckpt, cfg, device)
gnn.eval()

model_device = next(gnn.parameters()).device

# ---------------------------------------------------------
# Create a valid test graph
# Use the complete graph instead of taking 2 nodes
# with arbitrary edges.
# ---------------------------------------------------------
single = pyg_data.Data(
    x=b0.x,
    edge_index=b0.edge_index,
    edge_attr=b0.edge_attr
)

# One graph containing all nodes
single.batch = torch.zeros(
    single.x.shape[0],
    dtype=torch.long
)

# Move graph to the same device as the model
single = single.to(model_device)

# ---------------------------------------------------------
# Forward pass
# ---------------------------------------------------------
with torch.no_grad():
    pred, _ = gnn(single)

print(
    "smoke prediction:", pred.item(),
    "| device:", model_device,
    "| params:", sum(p.numel() for p in gnn.parameters())
)

In [ ]:
# RL helpers — imported from the committed rls/ package (Sep 2026 rewire).
# These were previously reimplemented inline in this cell; the inline copies
# drifted from rls/ (notably: no pairwise-symmetric masking, no
# topk_scheduled mode, pre-fix TTA), so this cell is imports-only now.
# Drift guard: tests/test_notebook_hygiene.py fails CI if inline copies return.
import torch
import torch.nn as nn
import torch.nn.functional as F

from rls.policy import EdgePolicyNet, build_policy
from rls.policy_gradient import (bernoulli_logp, bernoulli_entropy,
                                 sample_actions, compute_advantages,
                                 compute_pg_loss, PolicyGradientTrainer,
                                 ValueNet)
from rls.sparsify import (hard_mask, apply_min_keep_floor, repair_connectivity,
                          symmetrize_probs, repair_symmetric,
                          final_symmetric_mask, topk_scheduled_mask, eval_mask,
                          pair_asymmetry_fraction)
from rls.train_policy import (train_policy, prepare_graphs,
                              check_curriculum_divergence,
                              _graph_physics_terms as graph_physics_terms)
from rls.rewards import (compute_rewards, relative_virial_penalty,
                         virial_ratio_pruned, label_free_reward)
from rls.stageb import fine_tune_gnn, edge_dropout_masks
from rls.tta import adapt_at_test_time, mc_std, edge_kl, tta_should_enable
from rls.evaluate import build_results_table, save_paper_plots
from rls.provenance import (record_backbone, format_backbone_label,
                            require_backbone_label)


## Stage A: precompute embeddings, then train the RL policy

In [ ]:
# CELL 6: Precompute frozen node embeddings + graph contexts for ALL graphs
# (Sep 2026 rewire: uses rls.train_policy.prepare_graphs — the same adapter the
# offline trainer and scripts/multiseed_rl.py use. The hand-written prepare()
# lived here; its hasattr fallbacks are unnecessary on real TNG/CAMELS graphs,
# which always carry stellar_mass/vel_disp/pos.)
from torch_geometric.nn import global_mean_pool
import torch_geometric.data as pg
train_graphs = prepare_graphs(train_loader, gnn, device)
val_graphs = prepare_graphs(val_loader, gnn, device)
test_graphs = prepare_graphs(test_loader, gnn, device)
print("train graphs:", len(train_graphs), "| ctx dim:", train_graphs[0]["ctx"].shape)


In [ ]:
# CELL 7 (retired Sep 2026): ValueNet / compute_advantages / bernoulli_entropy
# now come from rls.policy_gradient (imported in the helpers cell above).
# This cell is intentionally empty — do not re-add inline copies
# (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 8 (retired Sep 2026): graph_physics_terms / relative_virial_penalty /
# compute_rewards now come from rls.train_policy and rls.rewards (imported in
# the helpers cell above). This cell is intentionally empty — do not re-add
# inline copies (tests/test_notebook_hygiene.py guards against drift).

In [ ]:
# CELL 9: GNN adapter — full and pruned predictions for a graph dict
# (same wiring as rls/run_experiment.py's gnns(); the old gnns_adapter lived here.)
from torch_geometric.data import Data, Batch
def gnns(graph, mask, use_gnn=gnn):
    with torch.no_grad():
        m = mask.to(device)
        assert m.dtype == torch.bool, f"expected bool mask, got {m.dtype}"
        d_full = Data(x=graph["x"], edge_index=graph["edge_index"],
                      edge_attr=graph["edge_attr"])
        pred_full, _ = gnn(Batch.from_data_list([d_full]))
        d_pr = Data(x=graph["x"], edge_index=graph["edge_index"][:, m],
                    edge_attr=graph["edge_attr"][m])
        pred_pruned, _ = gnn(Batch.from_data_list([d_pr]))
    return pred_full.view(-1), pred_pruned.view(-1)


In [ ]:
# CELL 10: Policy-gradient training loop (Stage A) — calls rls.train_policy.train_policy
# (Sep 2026 rewire: the hand-written REINFORCE loop lived here. It is replaced
# by the package call — same pattern as rls/run_experiment.py steps 3–7 — which
# adds pairwise-symmetric masking, topk_scheduled mode, and the
# curriculum-divergence guard. Accepted difference: the package iterates graphs
# in order (no per-epoch randperm shuffle), exactly like run_experiment.py.)
# HEADLINE PATH: sparsity_mode="topk_scheduled" (keep tracks the curriculum by
# construction). The old "penalty" path is kept as an appendix cell at the end
# for the paper's ablation table — headline artifacts below are topk only.
import pandas as pd
rls_cfg = cfg["rls"]                      # train_policy takes the SUB-dict
rls = rls_cfg                             # alias: later cells reference `rls`
rls_cfg["sparsity_mode"] = "topk_scheduled"
graphs = train_graphs                     # from CELL 6 (prepare_graphs)

policy = build_policy(cfg, node_emb_dim=gnn.output_dim).to(device)
value_net = ValueNet(gnn.output_dim).to(device)
opt = torch.optim.Adam(policy.parameters(), lr=rls_cfg["lr"])
vopt = torch.optim.Adam(value_net.parameters(), lr=rls_cfg["lr"])
trainer = PolicyGradientTrainer(policy, value_net, opt, vopt, rls_cfg)

log_rows = []
def log_fn(epoch, target_sp, loss):
    log_rows.append([epoch, target_sp, float(loss)])
    print(f"[epoch {epoch}] target_sparsity={target_sp:.3f} loss={float(loss):.4f}")
    if epoch % 10 == 0:  # keep the periodic artifact saves the old loop had
        torch.save(policy.state_dict(), "outputs/rls/policy.pt")
        torch.save(value_net.state_dict(), "outputs/rls/value_net.pt")

warn_rows = []
losses = train_policy(trainer, graphs, gnns, rls_cfg, device,
                      epochs=rls_cfg["epochs"], log_fn=log_fn,
                      warn_fn=lambda e, k, t: warn_rows.append((e, k, t)))
# Always save the FINAL weights (periodic saves stop at the last multiple of
# 10 — without this the checkpoint Notebooks C/D load would silently differ
# from the weights verified below).
torch.save(policy.state_dict(), "outputs/rls/policy.pt")
torch.save(value_net.state_dict(), "outputs/rls/value_net.pt")
print("Stage A done. Curriculum-divergence warnings:", warn_rows)
pd.DataFrame(log_rows, columns=["epoch", "target_sp", "loss"]).to_csv("outputs/rls/training_log.csv", index=False)


> **PITFALL NOTE (see plan Part 4):** if the `keep` column stays pinned at the `min_keep_frac` floor for 5+ epochs, the policy is collapsing — raise `w_conn` to 2.0 and restart. If `valMAE` explodes (>0.25), the sparsity curriculum is too aggressive — slow the anneal (change `sparsity_anneal_epochs` to 50).

## Stage B (optional): fine-tune the GNN on the policy's pruned graphs

Skip this if you plan to run Notebook D (TTA) next — TTA needs the untouched frozen backbone.

In [ ]:
# CELL 11: Stage B — fine-tune GNN on policy-pruned graphs (rls.stageb.fine_tune_gnn)
# (Sep 2026 rewire: the fixed 10-epoch loop regressed full-graph R2 0.9075 ->
# 0.8159 silently. Now val-guarded — best pruned-val epoch without full-val
# regression beyond stageb_full_tol, early-stop, best weights restored.
# Mirrors rls/run_experiment.py's Stage-B block, masks via eval_mask.)
_bb_frozen = record_backbone("frozen", gnn)
print("backbone before Stage B:", format_backbone_label(_bb_frozen))
ft_graphs = train_graphs[:]
masks = []
with torch.no_grad():
    for g in ft_graphs:
        p = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                 g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
        masks.append(eval_mask(g["edge_index"].to(device), p, rls_cfg))
val_masks = []
with torch.no_grad():
    for g in val_graphs:
        pv = torch.sigmoid(policy(g["edge_attr"].to(device), g["emb"].to(device),
                                  g["edge_index"].to(device), g["ctx"].to(device))).squeeze(-1)
        val_masks.append(eval_mask(g["edge_index"].to(device), pv, rls_cfg))
history, stageb_info = fine_tune_gnn(
    gnn, ft_graphs, masks, epochs=int(rls_cfg.get("stageb_epochs", 10)),
    lr=float(rls_cfg.get("stageb_lr", 1e-4)), device=device,
    val_graphs=val_graphs, val_masks=val_masks,
    patience=int(rls_cfg.get("stageb_patience", 3)),
    full_tol=float(rls_cfg.get("stageb_full_tol", 0.02)))
print(f"[stageB] best_epoch={stageb_info['best_epoch']} stopped_early={stageb_info['stopped_early']} "
      f"prefinetune_full={stageb_info['prefinetune_full']:.4f} best_full={stageb_info['best_full']} best_pruned={stageb_info['best_pruned']}")
torch.save(gnn.state_dict(), "outputs/rls/finetuned_gnn.pt")
print("Stage B done -> outputs/rls/finetuned_gnn.pt")
record_provenance("B_stageB", extra={
    "backbone_frozen": _bb_frozen,
    "backbone_finetuned": record_backbone("stageB_finetuned", gnn),
    "stageb": {k: v for k, v in stageb_info.items() if k in
               ("best_epoch", "stopped_early", "best_full", "best_pruned",
                "prefinetune_full")},
})


## Verify the trained policy on the test set

In [ ]:
# CELL 12: Verify final policy + fine-tuned GNN on test set
# (Sep 2026 rewire: RL mask via eval_mask so decoding matches the training
# sparsity_mode; the backbone state is printed with its provenance label —
# never report a "full graph" number without saying frozen vs stageB_finetuned.)
from model.physics_loss import MetricsComputer
_bb_verify = record_backbone("stageB_finetuned", gnn)
print("backbone:", format_backbone_label(_bb_verify), "(Stage B ran above in this notebook)")
preds_pol, preds_full, targets = [], [], []
with torch.no_grad():
    for b in test_loader:
        b = b.to(device)
        for i in range(b.num_graphs):
            g = b.get_example(i)
            n = g.x.shape[0]
            emb = gnn.get_embeddings(b, embedding_point="pre_pooling")
            ctx = global_mean_pool(emb, b.batch)
            p = torch.sigmoid(policy(g.edge_attr, emb[b.batch == i], g.edge_index, ctx[i])).squeeze(-1)
            m = eval_mask(g.edge_index, p, rls)
            d = pg.Data(x=g.x, edge_index=g.edge_index[:, m], edge_attr=g.edge_attr[m])
            d.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_pol, _ = gnn(d)
            preds_pol.append(p_pol.item()); targets.append(g.y.item())
            dfull = pg.Data(x=g.x, edge_index=g.edge_index, edge_attr=g.edge_attr)
            dfull.batch = torch.zeros(n, dtype=torch.long, device=device)
            p_full, _ = gnn(dfull)
            preds_full.append(p_full.item())
preds_pol = torch.tensor(preds_pol); preds_full = torch.tensor(preds_full); targets = torch.tensor(targets)
mp, mf = MetricsComputer.compute_all(preds_pol, targets), MetricsComputer.compute_all(preds_full, targets)
print(f"FULL  graph: RMSE={mf['rmse']:.4f} R2={mf['r2']:.4f}")
print(f"RL    pruned: RMSE={mp['rmse']:.4f} R2={mp['r2']:.4f}")
fid = np.corrcoef(preds_pol.numpy(), preds_full.numpy())[0, 1]
print(f"fidelity (Pearson): {fid:.4f}")


## Download your results

Grab `outputs/rls/policy.pt` (and `value_net.pt`) from Kaggle's output/session files, and add `policy.pt` to your `cosmicnet-data` Kaggle Dataset (Dataset -> Settings -> New Version -> upload) so Notebooks C and D can load it directly under `INPUT`. Also make sure Notebook A's `baselines.csv` is in that dataset — Notebook C's results table reads it, and it is not committed to the repo.

In [ ]:
# PROVENANCE (post-training): record the exact policy artifact B produced so
# C/D can verify they evaluate this very file (integrity check, not a new gate).
record_provenance("B_policy", extra={
    "policy_artifact": _file_info("outputs/rls/policy.pt"),
    "value_net_artifact": _file_info("outputs/rls/value_net.pt"),
})

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/rls_outputs", "zip", "outputs/rls")
print("Download /kaggle/working/rls_outputs.zip — contains policy.pt, value_net.pt, "
      "training_log.csv, and finetuned_gnn.pt if you ran Stage B")


In [ ]:
# APPENDIX (optional ablation): Stage A with the OLD sparsity_mode="penalty" path.
# Reproduces the diagnosed keep-everything collapse for the paper's ablation
# table — run ONLY if you need that row (extra ~20-30 min on T4). Headline
# artifacts above (policy.pt / value_net.pt / training_log.csv) are untouched;
# this variant saves to separate files.
rls_pen = dict(cfg["rls"], sparsity_mode="penalty")
policy_pen = build_policy(cfg, node_emb_dim=gnn.output_dim).to(device)
value_pen = ValueNet(gnn.output_dim).to(device)
opt_pen = torch.optim.Adam(policy_pen.parameters(), lr=rls_pen["lr"])
vopt_pen = torch.optim.Adam(value_pen.parameters(), lr=rls_pen["lr"])
trainer_pen = PolicyGradientTrainer(policy_pen, value_pen, opt_pen, vopt_pen, rls_pen)
log_pen, warn_pen = [], []
losses_pen = train_policy(trainer_pen, graphs, gnns, rls_pen, device,
                          epochs=rls_pen["epochs"],
                          log_fn=lambda e, t, l: (log_pen.append([e, t, float(l)]),
                                                  print(f"[penalty epoch {e}] target={t:.3f} loss={float(l):.4f}")),
                          warn_fn=lambda e, k, t: warn_pen.append((e, k, t)))
torch.save(policy_pen.state_dict(), "outputs/rls/policy_penalty.pt")
torch.save(value_pen.state_dict(), "outputs/rls/value_net_penalty.pt")
print("penalty ablation done. Divergence warnings (expect these — that IS the collapse):", warn_pen)
